In [ ]:
from google.colab import files

uploaded = files.upload()

Saving appointments.csv to appointments (1).csv


In [ ]:
import csv
import json

In [ ]:
#Task 1.1 — Read the file the foundations way

with open("appointments.csv", newline="") as file:
    reader = csv.DictReader(file)

    print("Header:", reader.fieldnames)

    rows = list(reader)

    print("Total rows:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [ ]:
def parse_row(row):
    row = row.copy()

    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [ ]:
accepted = []
rejected = []

for row in rows:
    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

Accepted rows: 988
Rejected rows: 32


In [ ]:
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")

Rejected rows saved successfully!


In [ ]:
files.download("rejected_rows.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Task 1.2 — Inspect and clean, pass 1

import pandas as pd

df = pd.read_csv("appointments.csv")


print("Shape:", df.shape)

print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Shape: (1020, 14)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id          

In [ ]:
df = df.drop_duplicates().copy()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [ ]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("Rows after cleaning booking_channel:", len(df))

Rows after cleaning booking_channel: 1000


In [ ]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Rows after date conversion:", len(df))

Rows after date conversion: 1000


In [ ]:
df = df.dropna(subset=["appt_date", "status"]).copy()

print("Rows after removing missing dates and status:", len(df))

Rows after removing missing dates and status: 990


In [ ]:
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

df.head()

Final Shape: (990, 14)

Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64

Duplicate Rows: 0


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-In,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-In,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended
5,A0005,P251,C01,2024-11-23,2025-01-02,64.0,F,19.0,700,Phone,Consultation,No,0,No-Show


In [ ]:
#Task 1.3 — Clean, pass 2: validity, gaps and outliers

import numpy as np

In [ ]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Missing age values:", df["age"].isnull().sum())

Missing age values: 17


In [ ]:
# Fill missing age with median
df["age"] = df["age"].fillna(df["age"].median())

# Round age and convert to integer
df["age"] = df["age"].round().astype(int)

# Fill missing distance with median
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

print("Missing ages:", df["age"].isnull().sum())
print("Missing distances:", df["distance_km"].isnull().sum())

Missing ages: 0
Missing distances: 0


In [ ]:
for col in ["distance_km", "fee_inr"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count values outside the bounds
    capped_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    # Cap outliers
    df[col] = np.clip(
        df[col],
        lower_bound,
        upper_bound
    )

    print(f"{col}: {capped_count} values capped")

distance_km: 26 values capped
fee_inr: 3 values capped


In [ ]:
# Round distance to 1 decimal place
df["distance_km"] = df["distance_km"].round(1)

# Round fee to whole rupees
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Rounding completed successfully!")

Rounding completed successfully!


In [ ]:
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nData Types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

df.head()

Final Shape: (990, 14)

Missing Values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Data Types:
age              int64
distance_km    float64
fee_inr          int64
dtype: object


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show


In [ ]:
#Task 1.4 — Features and a NumPy summary

# Days between booking and appointment
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Convert status into binary values
df["no_show"] = (df["status"] == "No-Show").astype(int)

df[["lead_days", "no_show"]].head()

,lead_days,no_show
0,1,1
1,10,0
2,6,0
4,4,0
5,40,1


In [ ]:
import numpy as np

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("Array Shape:", X.shape)

Array Shape: (990, 4)


In [ ]:
column_means = np.mean(X, axis=0)

print("Mean Age:", column_means[0])
print("Mean Distance:", column_means[1])
print("Mean Fee:", column_means[2])
print("Mean Lead Days:", column_means[3])

Mean Age: 48.173737373737374
Mean Distance: 9.50090909090909
Mean Fee: 635.7141414141414
Mean Lead Days: 11.819191919191919


In [ ]:
mask = X[:, 3] > 14

count = mask.sum()
share = mask.mean()

print("Appointments with lead_days > 14:", count)
print("Share:", share)
print("Percentage:", share * 100)

Appointments with lead_days > 14: 296
Share: 0.298989898989899
Percentage: 29.898989898989896


In [ ]:
distance = X[:, 1]

z_scores = (distance - np.mean(distance)) / np.std(distance)

outlier_count = (np.abs(z_scores) > 2).sum()

print("Number of rows with |z| > 2:", outlier_count)

Number of rows with |z| > 2: 53


In [ ]:
print(df[["lead_days", "no_show"]].head())

print("\nFinal DataFrame Shape:", df.shape)

   lead_days  no_show
0          1        1
1         10        0
2          6        0
4          4        0
5         40        1

Final DataFrame Shape: (990, 16)


In [ ]:
#Task 1.5 — Enrich the clinics from a live API
from google.colab import files

uploaded = files.upload()

Saving clinics.csv to clinics (1).csv


In [ ]:
import pandas as pd
import requests

clinics_df = pd.read_csv("clinics.csv")

print("Total Clinics:", len(clinics_df))
clinics_df.head()

Total Clinics: 8


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [ ]:
def get_temperature(clinic):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()

            temperatures = data.get("hourly", {}).get("temperature_2m")

            if not temperatures:
                reason = "Missing hourly temperature data"
                continue

            return temperatures[0]

        except (requests.exceptions.RequestException, ValueError) as e:
            reason = str(e)

    print(f"Failed: {clinic['name']} - {reason}")

    return None

In [ ]:
clinics_df["temp_c"] = clinics_df.apply(
    get_temperature,
    axis=1
)

print(clinics_df[["clinic_name", "temp_c"]])

                clinic_name  temp_c
0     Kothrud Family Clinic    16.1
1      Hadapsar Care Centre    18.5
2    Sitabuldi Health Point    20.8
3      Malviya Nagar Clinic    18.9
4     Vaishali Nagar Clinic    18.9
5  Edappally Medical Centre    25.3
6        Vijay Nagar Clinic    19.3
7    Gomti Nagar Care Point    19.0


In [ ]:
clinics_df.to_csv("clinics_enriched.csv", index=False)

print("clinics_enriched.csv saved successfully!")

clinics_enriched.csv saved successfully!


In [ ]:
files.download("clinics_enriched.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Task 1.6 — Export the clean table and write the SQL load script

df = df.sort_values("appointment_id").copy()

print("Total rows:", len(df))

Total rows: 990


In [ ]:
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

In [ ]:
columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

df = df[columns].copy()

df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [41]:
print("Appointment ID is unique:", df["appointment_id"].is_unique)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Appointment ID is unique: True

Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0


In [42]:
if not df["appointment_id"].is_unique:
    raise ValueError("Duplicate appointment IDs found!")

if df[columns].isnull().any().any():
    raise ValueError("Missing values found in appointments!")

df.to_csv("appointments_clean.csv", index=False)

print("appointments_clean.csv saved successfully!")

appointments_clean.csv saved successfully!


In [43]:
clinics_df = pd.read_csv("clinics_enriched.csv")

print("Total clinics:", len(clinics_df))
print("Clinic columns:", clinics_df.columns.tolist())

clinics_df.head()

Total clinics: 8
Clinic columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.1
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,18.9
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,18.9


In [44]:
def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        return f"'{value}'"

    return str(value)


with open("load_data.sql", "w") as file:

    # Insert clinics first
    for _, row in clinics_df.iterrows():

        values = ",".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO clinics VALUES ({values});\n"
        )

    # Insert appointments second
    for _, row in df.iterrows():

        values = ",".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO appointments VALUES ({values});\n"
        )

print("load_data.sql created successfully!")

load_data.sql created successfully!


In [45]:
with open("load_data.sql", "r") as file:
    statements = file.readlines()

print("Total SQL statements:", len(statements))

print("\nFirst SQL statement:")
print(statements[0])

print("\nFirst appointment statement:")
print(statements[len(clinics_df)])

print("\nLast SQL statement:")
print(statements[-1])

Total SQL statements: 998

First SQL statement:
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.1);


First appointment statement:
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);


Last SQL statement:
INSERT INTO appointments VALUES ('A1000','P085','C08','2024-12-03','2025-01-17',38,'M',10.8,550,'App','Follow-up','Yes',2,'No-Show',45,1);



In [46]:
from google.colab import files

files.download("appointments_clean.csv")
files.download("load_data.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
#Verify the results using Python (Google Colab)

clinic_summary = (
    df.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

clinic_summary = clinic_summary.merge(
    clinics_df[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="inner"
)

clinic_summary["no_show_pct"] = (
    100.0 * clinic_summary["no_shows"]
    / clinic_summary["appointments"]
).round(1)

clinic_summary = clinic_summary[
    ["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]
].sort_values("no_show_pct", ascending=False)

print(clinic_summary.to_string(index=False))

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8


In [48]:
highest = clinic_summary.iloc[0]

print("Clinic:", highest["clinic_name"])
print("City:", highest["city"])
print("Appointments:", highest["appointments"])
print("No-Shows:", highest["no_shows"])
print("No-Show Rate:", highest["no_show_pct"], "%")

Clinic: Gomti Nagar Care Point
City: Lucknow
Appointments: 90
No-Shows: 30
No-Show Rate: 33.3 %


In [49]:
print("Total Appointments:", clinic_summary["appointments"].sum())
print("Total No-Shows:", clinic_summary["no_shows"].sum())

Total Appointments: 990
Total No-Shows: 218


In [50]:
#Task 3.1 — Features, Split and Baseline

from google.colab import files

uploaded = files.upload()

Saving appointments_clean.csv to appointments_clean (1).csv


In [52]:
#Import libraries and load the dataset

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv("appointments_clean.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (990, 16)


In [53]:
#Select features and target

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

y = df["no_show"]

In [54]:
#Convert reminder_sent to 1/0

X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

print(X["reminder_sent"].value_counts())

reminder_sent
1    561
0    429
Name: count, dtype: int64


In [55]:
#One-hot encode categorical features

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

print("Features:", X.columns.tolist())
print("Feature Shape:", X.shape)

Features: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Feature Shape: (990, 11)


In [56]:
#Split training and test data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Size:", len(X_train))
print("Test Size:", len(X_test))

print("No-Shows in Test Set:", y_test.sum())

Training Size: 792
Test Size: 198
No-Shows in Test Set: 49


In [57]:
#Calculate baseline accuracy

baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Accuracy:", round(baseline_accuracy, 4))

print("Baseline Accuracy (%):",
      round(baseline_accuracy * 100, 2))

Baseline Accuracy: 0.7525
Baseline Accuracy (%): 75.25


In [58]:
#Final verification

print("Training Rows:", X_train.shape[0])
print("Test Rows:", X_test.shape[0])
print("Test No-Shows:", int(y_test.sum()))
print("Baseline Accuracy:", round(baseline_accuracy * 100, 2), "%")

Training Rows: 792
Test Rows: 198
Test No-Shows: 49
Baseline Accuracy: 75.25 %


In [59]:
# Logistic regression

#Import required libraries

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np
import pandas as pd

In [60]:
#Scale the features

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training Shape:", X_train_s.shape)
print("Test Shape:", X_test_s.shape)

Training Shape: (792, 11)
Test Shape: (198, 11)


In [61]:
#Train the logistic regression model

model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [62]:
#Make predictions

y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]

In [63]:
#Evaluate model performance

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889


In [64]:
#Print coefficients with feature names

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients.to_string(index=False))

                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658


In [65]:
#Calculate reminder odds ratios

reminder_index = X.columns.get_loc("reminder_sent")

reminder_coef = model.coef_[0][reminder_index]

scaled_odds_ratio = np.exp(reminder_coef)

print("Reminder Coefficient:", reminder_coef)
print("Scaled Odds Ratio:", scaled_odds_ratio)

Reminder Coefficient: -0.5806686033659922
Scaled Odds Ratio: 0.559524141750829


In [66]:
#Calculate the odds ratio for changing reminder_sent from No (0) to Yes (1).

reminder_std = scaler.scale_[reminder_index]

original_coef = reminder_coef / reminder_std

actual_odds_ratio = np.exp(original_coef)

print("Reminder Standard Deviation:", reminder_std)
print("Original-Scale Coefficient:", original_coef)
print("No-to-Yes Odds Ratio:", actual_odds_ratio)

Reminder Standard Deviation: 0.4949752621616814
Original-Scale Coefficient: -1.1731265130909097
No-to-Yes Odds Ratio: 0.3093980903124433


In [ ]:
#Explain the difference

"""The two odds ratios differ because the model was trained on standardized features:
the first represents a one-standard-deviation increase,
while the second represents changing the reminder value from 0 to 1."""


In [67]:
#Write the conclusion

if actual_odds_ratio < 1:
    direction = "lower"
elif actual_odds_ratio > 1:
    direction = "higher"
else:
    direction = "equal"

print(
    f"In this model, holding the other features constant, "
    f"receiving a reminder is associated with {direction} "
    f"odds of a no-show (odds ratio = {actual_odds_ratio:.3f}). "
    f"This is an observational association, not proof of causation."
)

In this model, holding the other features constant, receiving a reminder is associated with lower odds of a no-show (odds ratio = 0.309). This is an observational association, not proof of causation.


In [68]:
#Task 3.3 — Decision tree and overfitting

#Import required libraries

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [69]:
#Train a decision tree without a depth limit

tree_full = DecisionTreeClassifier(random_state=42)

tree_full.fit(X_train, y_train)

train_pred_full = tree_full.predict(X_train)
test_pred_full = tree_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)
test_acc_full = accuracy_score(y_test, test_pred_full)

print("Decision Tree (No Depth Limit)")
print("Training Accuracy:", round(train_acc_full, 4))
print("Test Accuracy:", round(test_acc_full, 4))

Decision Tree (No Depth Limit)
Training Accuracy: 1.0
Test Accuracy: 0.6717


In [70]:
#Train a decision tree with max_depth=3

tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train, y_train)

train_pred_depth3 = tree_depth3.predict(X_train)
test_pred_depth3 = tree_depth3.predict(X_test)

train_acc_depth3 = accuracy_score(y_train, train_pred_depth3)
test_acc_depth3 = accuracy_score(y_test, test_pred_depth3)

print("Decision Tree (Max Depth = 3)")
print("Training Accuracy:", round(train_acc_depth3, 4))
print("Test Accuracy:", round(test_acc_depth3, 4))

Decision Tree (Max Depth = 3)
Training Accuracy: 0.822
Test Accuracy: 0.7778


In [71]:
#Compare both models

results = pd.DataFrame({
    "Model": ["No Depth Limit", "Max Depth = 3"],
    "Training Accuracy": [train_acc_full, train_acc_depth3],
    "Test Accuracy": [test_acc_full, test_acc_depth3],
    "Accuracy Gap": [
        train_acc_full - test_acc_full,
        train_acc_depth3 - test_acc_depth3
    ]
})

print(results.round(4).to_string(index=False))

         Model  Training Accuracy  Test Accuracy  Accuracy Gap
No Depth Limit              1.000         0.6717        0.3283
 Max Depth = 3              0.822         0.7778        0.0442


In [72]:
#Explain overfitting

print(
    "A larger gap between training and test accuracy indicates "
    "overfitting, where the model learns the training data too "
    "closely and performs less effectively on unseen data."
)

A larger gap between training and test accuracy indicates overfitting, where the model learns the training data too closely and performs less effectively on unseen data.


In [73]:
#Task 3.4 — Random forest and a comparison table

#Import required libraries

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import pandas as pd

In [74]:
#Train the Random Forest model

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [75]:
#Evaluate Random Forest performance

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Results")
print("Accuracy:", round(rf_accuracy, 4))
print("Recall:", round(rf_recall, 4))
print("ROC-AUC:", round(rf_auc, 4))

Random Forest Results
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057


In [76]:
#Find the top 3 important features

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top 3 Important Features:")
print(feature_importance.head(3).to_string(index=False))

Top 3 Important Features:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739


In [77]:
#Evaluate the depth-3 decision tree

tree_pred = tree_depth3.predict(X_test)
tree_prob = tree_depth3.predict_proba(X_test)[:, 1]

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_recall = recall_score(y_test, tree_pred)
tree_auc = roc_auc_score(y_test, tree_prob)

In [78]:
#Create the model comparison table

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        tree_accuracy,
        rf_accuracy
    ],
    "Recall": [
        recall,
        tree_recall,
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc,
        tree_auc,
        rf_auc
    ]
})

print(comparison.round(4).to_string(index=False))

                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (Depth 3)    0.7778  0.2245   0.6997
          Random Forest    0.7475  0.3061   0.7057


In [79]:
#Identify the best-looking model

best_model = comparison.sort_values(
    by=["ROC-AUC", "Recall"],
    ascending=False
).iloc[0]

print("Best-looking model:", best_model["Model"])

print(
    f"{best_model['Model']} has the highest ROC-AUC "
    f"({best_model['ROC-AUC']:.4f}) in this comparison, "
    "with recall used as a secondary consideration. "
    "ROC-AUC measures how well the model ranks no-show risk, "
    "while recall measures how many actual no-shows it detects."
)

print(
    "This is a descriptive test-set comparison only. "
    "Final model selection must use training data and "
    "cross-validation in Task 3.5."
)

Best-looking model: Logistic Regression
Logistic Regression has the highest ROC-AUC (0.7889) in this comparison, with recall used as a secondary consideration. ROC-AUC measures how well the model ranks no-show risk, while recall measures how many actual no-shows it detects.
This is a descriptive test-set comparison only. Final model selection must use training data and cross-validation in Task 3.5.


In [80]:
#Task 3.5 — Tune, choose the model and the threshold, then test once

#Import required libraries

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


In [81]:
#Tune Random Forest using GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673


In [82]:
#Compare models using training cross-validation

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

lr_cv = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv = cross_val_score(
    tree, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

forest_cv = cross_val_score(
    forest, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_forest_cv = grid.best_score_

cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv,
        tree_cv,
        forest_cv,
        tuned_forest_cv
    ]
})

print(cv_results.round(4).to_string(index=False))

                  Model  CV ROC-AUC
    Logistic Regression      0.7737
Decision Tree (Depth 3)      0.6784
          Random Forest      0.7466
    Tuned Random Forest      0.7673


In [83]:
#Select the best model using CV ROC-AUC

models = {
    "Logistic Regression": lr_pipe,
    "Decision Tree (Depth 3)": tree,
    "Random Forest": forest,
    "Tuned Random Forest": grid.best_estimator_
}

best_model_name = cv_results.loc[
    cv_results["CV ROC-AUC"].idxmax(),
    "Model"
]

final_model = models[best_model_name]

print("Selected Model:", best_model_name)
print("Selection Basis: Highest training CV ROC-AUC")

Selected Model: Logistic Regression
Selection Basis: Highest training CV ROC-AUC


In [84]:
#Generate out-of-fold probabilities

oof_probs = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Out-of-fold probabilities generated!")

Out-of-fold probabilities generated!


In [85]:
#Compare thresholds 0.50 and 0.35

threshold_results = []

for threshold in [0.50, 0.35]:

    predictions = (oof_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_train, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_train, predictions, zero_division=0
        ),
        "Number Flagged": int(predictions.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Precision  Recall     F1  Number Flagged
      0.50     0.5753  0.2485 0.3471              73
      0.35     0.5130  0.4675 0.4892             154


In [86]:
#Choose threshold with higher F1

best_threshold = threshold_df.loc[
    threshold_df["F1"].idxmax(),
    "Threshold"
]

print("Selected Threshold:", best_threshold)
print("Reason: Higher out-of-fold training F1 score.")

Selected Threshold: 0.35
Reason: Higher out-of-fold training F1 score.


In [87]:
#Fit the selected model on training data

final_model.fit(X_train, y_train)

print("Final model trained successfully!")

Final model trained successfully!


In [88]:
#Evaluate once on the test set

test_probs = final_model.predict_proba(X_test)[:, 1]

test_flags = (test_probs >= best_threshold).astype(int)

test_precision = precision_score(
    y_test, test_flags, zero_division=0
)

test_recall = recall_score(
    y_test, test_flags, zero_division=0
)

test_auc = roc_auc_score(y_test, test_probs)

print("Final Model:", best_model_name)
print("Selected Threshold:", best_threshold)

print("Test Precision:", round(test_precision, 4))
print("Test Recall:", round(test_recall, 4))
print("Number Flagged:", int(test_flags.sum()))
print("Test ROC-AUC:", round(test_auc, 4))

Final Model: Logistic Regression
Selected Threshold: 0.35
Test Precision: 0.5714
Test Recall: 0.5714
Number Flagged: 49
Test ROC-AUC: 0.7889


In [89]:
#Create test_risk_scores.csv

test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.to_numpy(),
    "risk_prob": test_probs,
    "flag": test_flags
})

print("Shape:", test_risk_scores.shape)

test_risk_scores.head()

Shape: (198, 4)


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.021565,0
1,A0336,0,0.150917,0
2,A0508,0,0.518692,1
3,A0314,0,0.191300,0
4,A0089,0,0.167718,0


In [90]:
#Save and download the CSV

test_risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

from google.colab import files

files.download("test_risk_scores.csv")

print("test_risk_scores.csv saved successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

test_risk_scores.csv saved successfully!


In [93]:
#Write the business conclusion
print("Business conclusion:")
print("In a real clinic, the risk threshold should be selected based on how many follow-up calls the front desk can make each evening, balanced against the financial cost of an empty appointment slot.")

Business conclusion:
In a real clinic, the risk threshold should be selected based on how many follow-up calls the front desk can make each evening, balanced against the financial cost of an empty appointment slot.


In [95]:
#GenAI: turn call-centre notes into structured reasons, then recommend

#Task 4.1 — One Validated Gemini API Call

#Install required libraries

!pip install -q google-genai pydantic


In [122]:
#Import libraries and connect Gemini

import json
import pandas as pd
from getpass import getpass

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

API_KEY = getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

# Updated to the supported gemini-3.8-flash model
MODEL_NAME = "gemini-3.5-flash-lite"

Paste your Gemini API key: ··········


In [123]:
#Upload followup_notes.csv

from google.colab import files

uploaded = files.upload()

notes_df = pd.read_csv("followup_notes.csv")

print("Total notes:", len(notes_df))
notes_df.head()

Saving followup_notes.csv to followup_notes (3).csv
Total notes: 30


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [124]:
#Define the JSON response schema

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [125]:
#Create the system instruction

SYSTEM_TEXT = """
You are an assistant analysing clinic appointment no-show notes.

Use only the information provided in the note.
Do not invent patient details.
Identify the most likely root cause.
Return valid JSON matching the supplied schema.
Keep the summary within 25 words.
Use a professional, neutral tone.
"""

In [126]:
#Create the analyse_note() function

def analyse_note(note_text):

    prompt = f"""
    Context:
    CareLoop Clinics wants to understand why patients miss
    their scheduled appointments.

    Role:
    You are a healthcare operations analyst.

    Action:
    Analyse the following call-centre note and identify
    the most likely reason for the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    Choose exactly one root cause.
    Provide confidence between 0 and 1.
    Write a summary of no more than 25 words.
    Suggest one practical follow-up action.

    Tone:
    Professional, concise and neutral.

    Format:
    Return JSON containing exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    Call-centre note:
    {note_text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    data = json.loads(response.text)

    return data

In [129]:
MODEL_NAME = "gemini-3.5-flash-lite"

note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result = analyse_note(note_n001)

print("Gemini Response:")
print(json.dumps(result, indent=2))

Gemini Response:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "Patient missed the appointment because she did not receive a reminder and forgot the visit. She requested to rebook.",
  "recommended_action": "Reschedule the appointment for next week and verify the patient's contact details for future reminders."
}


In [130]:
#Validate Gemini Response with Pydantic

from pydantic import BaseModel, Field

class NoteAnalysis(BaseModel):
    root_cause: str
    confidence: float = Field(ge=0, le=1)
    summary: str
    recommended_action: str

# Validate Gemini response
validated = NoteAnalysis.model_validate(result)

# Check summary length
assert len(validated.summary.split()) <= 25

print("Validation successful!")
print(validated.model_dump())

Validation successful!
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient missed the appointment because she did not receive a reminder and forgot the visit. She requested to rebook.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's contact details for future reminders."}


In [131]:
#Task 4.2 — Validate and process the 10 notes

#Import required libraries

import pandas as pd
import json
import time

from pydantic import BaseModel, Field, ValidationError

In [132]:
#Define the Pydantic model

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

In [133]:
#Define allowed root causes

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

def validate_root_cause(root_cause):
    return root_cause in ALLOWED_CAUSES

In [134]:
#Load the first 10 notes

notes_df = pd.read_csv("followup_notes.csv")

first_10_notes = notes_df.head(10)

print("Total Notes:", len(notes_df))
print("Notes to Analyse:", len(first_10_notes))

Total Notes: 30
Notes to Analyse: 10


In [135]:
#Process and validate all 10 notes

results = []

for _, row in first_10_notes.iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"Processing {note_id}...")

    # Check for blank or missing notes
    if pd.isna(note_text) or str(note_text).strip() == "":

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

        print(f"{note_id}: Skipped blank note")
        continue

    raw_response = ""
    data = {}

    try:
        # Call Gemini and preserve its JSON response
        data = analyse_note(note_text)

        raw_response = json.dumps(data, ensure_ascii=False)

        # Validate with Pydantic
        validated = FollowupInsight(**data)

        # Check allowed root causes
        if not validate_root_cause(validated.root_cause):
            raise ValueError("Invalid root_cause")

        validation_status = "valid"

    except (ValidationError, ValueError, TypeError) as e:
        validation_status = "invalid"
        print(f"{note_id}: Validation failed - {e}")

    except Exception as e:
        validation_status = "invalid"
        print(f"{note_id}: API error - {e}")

    # Keep whatever fields the model returned
    results.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": data.get("root_cause"),
        "confidence": data.get("confidence"),
        "summary": data.get("summary"),
        "recommended_action": data.get("recommended_action"),
        "validation_status": validation_status,
        "raw_response": raw_response
    })

    # Pause between API calls
    time.sleep(13)

print("Processing completed!")

Processing N001...
N001: API error - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N002...
N002: API error - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N003...
N003: API error - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N004...
N004: Skipped blank note
Processing N005...
Processing N006...
Processing N007...
Processing N008...
N008: API error - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.

In [136]:
#Create the results DataFrame

notes_analysed = pd.DataFrame(
    results,
    columns=[
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
)

print("Output Shape:", notes_analysed.shape)

notes_analysed.head(10)

Output Shape: (10, 8)


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,None,NaN,None,None,invalid,
1,N002,A0897,None,NaN,None,None,invalid,
2,N003,A0526,None,NaN,None,None,invalid,
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.95,Patient missed the appointment due to acute il...,Contact the patient for rescheduling once they...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid..."
5,N006,A0507,work_or_schedule,1.00,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening slot.,valid,"{""root_cause"": ""work_or_schedule"", ""confidence..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because they di...,Reschedule the appointment and verify the pati...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi..."
7,N008,A0222,None,NaN,None,None,invalid,
8,N009,A0977,other,0.80,Patient unreachable as phone was switched off ...,Send a written communication to the patient re...,valid,"{""root_cause"": ""other"", ""confidence"": 0.8, ""su..."
9,N010,A0596,None,NaN,None,None,invalid,


In [137]:
#Check validation status

print(notes_analysed["validation_status"].value_counts())

print("\nRoot Cause Counts:")
print(notes_analysed["root_cause"].value_counts())

validation_status
invalid          5
valid            4
skipped_blank    1
Name: count, dtype: int64

Root Cause Counts:
root_cause
other                    2
illness_or_emergency     1
work_or_schedule         1
forgot_or_no_reminder    1
Name: count, dtype: int64


In [138]:
#Save notes_analysed.csv

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("notes_analysed.csv saved successfully!")

notes_analysed.csv saved successfully!


In [139]:
#Download the file

from google.colab import files

files.download("notes_analysed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [140]:
#Task 4.3 — Break it on purpose

#Test two deliberately invalid responses

from pydantic import ValidationError

# Test 1: Invalid confidence
bad_reply_1 = {
    "root_cause": "cost_or_fee",
    "confidence": 1.7,
    "summary": "Patient could not afford the fee",
    "recommended_action": "Offer payment assistance"
}

# Test 2: Missing recommended_action
bad_reply_2 = {
    "root_cause": "work_or_schedule",
    "confidence": 0.8,
    "summary": "Patient had a work conflict"
}

for name, reply in [
    ("Invalid Confidence", bad_reply_1),
    ("Missing Action", bad_reply_2)
]:
    try:
        FollowupInsight(**reply)
        print(name, ": Valid")

    except ValidationError as e:
        print(f"\n{name}:")
        print(e)


Invalid Confidence:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing Action:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...nt had a work conflict'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [141]:
#Create two prompts for comparison

def create_prompt(note_text, version):

    if version == "V1":
        return f"Classify this note:\n{note_text}"

    return f"""
    Context:
    CareLoop Clinics wants to understand why patients
    missed their appointments.

    Role:
    You are a healthcare operations analyst.

    Action:
    Identify the most likely root cause of the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    If the reason is ambiguous or unclear,
    choose "other" rather than guessing.

    Tone:
    Professional, neutral and concise.

    Format:
    Return JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.

    Confidence must be between 0 and 1.
    Summary must contain no more than 25 words.

    Note:
    {note_text}
    """

In [142]:
#Create a function to test both prompts

def test_prompt(note_text, version):

    prompt = create_prompt(note_text, version)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)

In [143]:
#Load notes N007 and N009

notes_df = pd.read_csv("followup_notes.csv")

test_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]

print(test_notes[["note_id", "note_text"]])

  note_id                                          note_text
6    N007  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
8    N009  Phone was switched off, could not reach the pa...


In [144]:
#Run both prompts on both notes

import time

comparison_results = []

api_calls_43 = 0

for _, row in test_notes.iterrows():

    for version in ["V1", "V2"]:

        try:
            result = test_prompt(row["note_text"], version)
            api_calls_43 += 1

            root_cause = result.get("root_cause")

            comparison_results.append({
                "note_id": row["note_id"],
                "prompt": version,
                "root_cause": root_cause,
                "confidence": result.get("confidence"),
                "allowed_value": root_cause in ALLOWED_CAUSES
            })

        except Exception as e:
            api_calls_43 += 1

            print(f"Error for {row['note_id']} ({version}): {e}")

            comparison_results.append({
                "note_id": row["note_id"],
                "prompt": version,
                "root_cause": None,
                "confidence": None,
                "allowed_value": False
            })

        time.sleep(13)

In [145]:
#Display the comparison table

comparison_df = pd.DataFrame(comparison_results)

print(comparison_df.to_string(index=False))

note_id prompt                    root_cause  confidence  allowed_value
   N007     V1 forgot_appointment_and_no_sms        0.95          False
   N007     V2         forgot_or_no_reminder        0.95           True
   N009     V1           Patient unreachable        1.00          False
   N009     V2                         other        0.50           True


In [147]:
#Choose the prompt for deployment

"""Conclusion: I would deploy V2 (CRAFT) because it provides clear instructions, restricts the root-cause categories and tells the model to select "other" when the reason is ambiguous. This reduces unsupported guesses and makes the output easier to validate."""

'Conclusion: I would deploy V2 (CRAFT) because it provides clear instructions, restricts the root-cause categories and tells the model to select "other" when the reason is ambiguous. This reduces unsupported guesses and makes the output easier to validate.'

In [148]:
#Count total API calls across Tasks 4.1–4.3

# Adjust these counts to your actual API usage.
api_calls_41 = 1
api_calls_42 = 9

total_api_calls = (
    api_calls_41 +
    api_calls_42 +
    api_calls_43
)

print("Task 4.1 API Calls:", api_calls_41)
print("Task 4.2 API Calls:", api_calls_42)
print("Task 4.3 API Calls:", api_calls_43)

print("Total API Calls:", total_api_calls)

Task 4.1 API Calls: 1
Task 4.2 API Calls: 9
Task 4.3 API Calls: 4
Total API Calls: 14


In [149]:
#Task 4.4 — Recommendation and GitHub

#Load and merge both CSV files

import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("notes_analysed.csv")

merged = pd.merge(
    notes,
    appointments,
    on="appointment_id",
    how="left"
)

print("Merged Shape:", merged.shape)
merged.head()

Merged Shape: (10, 23)


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,NaN,NaN,NaN,NaN,invalid,NaN,P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,NaN,NaN,NaN,NaN,invalid,NaN,P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,NaN,NaN,NaN,NaN,invalid,NaN,P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness_or_emergency,0.95,Patient missed the appointment due to acute il...,Contact the patient for rescheduling once they...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1


In [150]:
#Cross-tabulate root cause against reminders

cross_tab = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)

print("Root Cause vs Reminder Sent:")
print(cross_tab)

Root Cause vs Reminder Sent:
reminder_sent          No  Yes
root_cause                    
forgot_or_no_reminder   1    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        0    1


In [151]:
#Calculate numbers for the README

print("Total Appointments:", len(appointments))
print("Total No-Shows:", appointments["no_show"].sum())

print("Overall No-Show Rate:",
      round(appointments["no_show"].mean() * 100, 2), "%")

print("Notes Analysed:", len(notes))

print("\nRoot Cause Counts:")
print(notes["root_cause"].value_counts())

print("\nValidation Status:")
print(notes["validation_status"].value_counts())

print("\nFinal Model:", best_model_name)
print("Threshold:", best_threshold)
print("Test ROC-AUC:", round(test_auc, 4))
print("Test Recall:", round(test_recall, 4))
print("Test Precision:", round(test_precision, 4))

Total Appointments: 990
Total No-Shows: 218
Overall No-Show Rate: 22.02 %
Notes Analysed: 10

Root Cause Counts:
root_cause
other                    2
illness_or_emergency     1
work_or_schedule         1
forgot_or_no_reminder    1
Name: count, dtype: int64

Validation Status:
validation_status
invalid          5
valid            4
skipped_blank    1
Name: count, dtype: int64

Final Model: Logistic Regression
Threshold: 0.35
Test ROC-AUC: 0.7889
Test Recall: 0.5714
Test Precision: 0.5714
